# Phase 4 — Session-level SMI Comparisons (DREADD saline/DCZ cohort)

Track B only for now (population-level, no cross-session cell tracking) —
per the roadmap: Track A (tracked-cell, paired within-cell) versions of
this phase, Phase 5 (layer-specific), and Phase 6 (landmark preference)
all get revisited together after Phase 6, reusing Phase 3's already-saved
`*_smi_results_dreadd.h5` per session.

Planned comparisons (unpaired across sessions, since Track B doesn't track
cell identity):
- `SalineDCZ_1/2/3` — each within-day saline vs. DCZ pair
- `Baseline_vs_DCZ` — baseline (Day5) vs. DCZ sessions
- `OpenLoopActive` — active open-loop saline vs. DCZ
- `OpenLoopStationary` — stationary open-loop saline vs. DCZ

Built incrementally, one function at a time. Consolidated into
`4.SessionComparison.py` only once everything here works end-to-end on
real data.

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")

import os
import re
import glob
import numpy as np
import h5py
import pandas as pd
import matplotlib
matplotlib.use('Qt5Agg')
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.widgets import CheckButtons, Button
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations

rcParams['legend.fontsize'] = 20
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 25
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20

# Existing pipeline code, reused via import -- not modified. Needed for
# Function 4.6's reliability-component recomputation (pattern_reliable /
# active_cells), same functions Phase 3's Function 3.3b already used.
from helper import files
from helper.ReliabilityTesting import evaluate_pattern_similarity_improved, improved_activity_threshold_check

# One real DREADD session's already-saved Phase 3 output, to develop and
# sanity-check Function 4.1 against -- kept for BOTH animals so switching
# which one is "active" can't silently leave TEST_SESSION_DIR/
# TEST_SMI_SAVE_PATH pointing at the other animal's Day1 folder (exactly
# the mismatch that broke this cell before: one animal's dir combined with
# the other animal's session subfolder name, which doesn't exist).
TEST_ANIMAL_DIR_JSY090 = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD"
TEST_ANIMAL_DIR_JSY093 = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD"
TEST_SESSION_DIR_JSY090 = os.path.join(TEST_ANIMAL_DIR_JSY090,
                                        "260719_JSY_JSY090_LongitudinalImaging_DREADD_Day1",
                                        "TSeries-07192026-0941-001")
TEST_SESSION_DIR_JSY093 = os.path.join(TEST_ANIMAL_DIR_JSY093,
                                        "260719_JSY_JSY093_LongitudinalImaging_DREADD_Day1",
                                        "TSeries-07192026-0941-001")

# --- Pick which animal is "active" for the rest of this notebook -- just
# swap which pair these three point at to switch animals. ---
TEST_ANIMAL_DIR = TEST_ANIMAL_DIR_JSY090
TEST_SESSION_DIR = TEST_SESSION_DIR_JSY090
TEST_SMI_SAVE_PATH = os.path.join(TEST_SESSION_DIR, os.path.basename(TEST_SESSION_DIR) + '_smi_results_dreadd.h5')

## Function 4.1 — `load_session_smi_for_comparison`

Reads one session's already-saved `*_smi_results_dreadd.h5` (Function 3.2's
output) directly from disk into a tidy per-cell DataFrame — the shared
building block every Phase 4/5/6 comparison starts from. Reading from disk
rather than reusing an in-memory `run_smi_analysis_session` result
decouples this phase from Phase 3's kernel session, and means Track A's
later revisit can reuse this same loader unchanged.

- **Input:** `save_path` (a `*_smi_results_dreadd.h5` file),
  `condition_label` (e.g. `'saline'`/`'dcz'` — how Function 4.2 will group
  rows across sessions), `session_label` (optional override; defaults to
  the h5's stored attr).
- **Output:** `df` — one row per cell: `cell_idx`, `SMI`, `valid`,
  `analysis_reliable`, `combined_reliable`, `avg_cc`, `layer`, `condition`,
  `session_label`.

In [2]:
def load_session_smi_for_comparison(save_path, condition_label=None, session_label=None):
    """
    Load one session's already-computed *_smi_results_dreadd.h5 into a tidy
    per-cell DataFrame. See markdown above.

    Parameters
    ----------
    save_path : str
        Path to a *_smi_results_dreadd.h5 file (Function 3.2's output).
    condition_label : str, optional
        Tag stored in a 'condition' column. Leave None to fill in later
        (e.g. via build_comparison_table).
    session_label : str, optional
        Tag stored in a 'session_label' column. Defaults to the h5's
        stored session_label attr.

    Returns
    -------
    df : pandas.DataFrame
    """
    with h5py.File(save_path, 'r') as f:
        stored_label = f.attrs.get('session_label', os.path.basename(save_path))
        SMI_values = f['global_smi/SMI_all_cells'][:]
        valid_cells_mask = f['global_smi/valid_cells_mask'][:]
        analysis_reliable_cells = f['global_smi/analysis_reliable_cells'][:]
        combined_reliable = f['reliability/combined_reliable'][:]
        avg_cc = f['reliability/avg_cc'][:]
        cohen_d = f['reliability/cohen_d'][:]

        n_cells = len(SMI_values)
        layer_of_cell = np.full(n_cells, None, dtype=object)
        for safe_name in f['layer_smi']:
            layer_grp = f['layer_smi'][safe_name]
            original_name = layer_grp.attrs.get('original_name', safe_name)
            cell_indices = layer_grp['cell_indices'][:]
            layer_of_cell[cell_indices] = original_name

    df = pd.DataFrame({
        'cell_idx': np.arange(n_cells),
        'SMI': SMI_values,
        'valid': valid_cells_mask,
        'analysis_reliable': analysis_reliable_cells,
        'combined_reliable': combined_reliable,
        'avg_cc': avg_cc,
        'cohen_d': cohen_d,
        'layer': layer_of_cell,
    })
    df['condition'] = condition_label
    df['session_label'] = session_label if session_label is not None else stored_label

    print(f"Loaded {n_cells} cells from {save_path}")
    print(f"  analysis_reliable: {df['analysis_reliable'].sum()}, valid: {df['valid'].sum()}")

    return df

In [3]:
# --- Try it on the real Day1 session's already-saved SMI output ---
day1_df = load_session_smi_for_comparison(TEST_SMI_SAVE_PATH, condition_label='baseline', session_label='Day1')
day1_df.head(10)

Loaded 1048 cells from D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260719_JSY_JSY093_LongitudinalImaging_DREADD_Day1\TSeries-07192026-0941-001\TSeries-07192026-0941-001_smi_results_dreadd.h5
  analysis_reliable: 280, valid: 280


,cell_idx,SMI,valid,analysis_reliable,combined_reliable,avg_cc,cohen_d,layer,condition,session_label
0,0,0.930589,False,False,False,0.168647,1.698071,L6,baseline,Day1
1,1,-0.323651,False,False,False,0.000000,0.000000,L5,baseline,Day1
2,2,0.013192,False,False,False,0.181977,2.724047,L2/3,baseline,Day1
3,3,-0.012781,False,False,False,0.000000,0.000000,L5,baseline,Day1
4,4,0.688246,False,False,False,0.196448,3.127496,L5,baseline,Day1
5,5,0.829353,True,True,True,0.309171,4.892438,L5,baseline,Day1
6,6,0.628644,False,False,False,0.061170,1.211927,L6,baseline,Day1
7,7,0.767609,False,False,False,0.038356,0.364433,L5,baseline,Day1
8,8,0.220549,False,False,False,0.420032,5.774504,L6,baseline,Day1
9,9,-0.773526,False,False,False,0.000000,0.000000,L6,baseline,Day1


## Function 4.2 — `discover_smi_sessions` + `select_sessions_popup`

Same checkbox-picker workflow as Phase 3's Function 3.5, but scanning for
already-computed `*_smi_results_dreadd.h5` files instead of raw
`suite2p/plane0` folders — Phase 4 only ever works from Phase 3's saved
output, never recomputes SMI itself.

- **`discover_smi_sessions`** — reimplemented with the same naming/
  disambiguation logic as `discover_animal_sessions`, just pointed at
  `*_smi_results_dreadd.h5` files. Catalog entries store `save_path`
  instead of `plane0_path`.
- **`select_sessions_popup`** — copied over completely unchanged from
  Phase 3; it only ever reads `session_catalog[label]['session_type']`,
  so it doesn't care what kind of path each entry actually points to.

- **Input:** `animal_dir` (for discovery); `session_catalog` (for the
  picker).
- **Output:** `catalog` (`{label: {'save_path', 'session_type',
  'tseries_dir'}}`); `selected_labels` (list of str).

In [4]:
def discover_smi_sessions(animal_dir):
    """
    Scan animal_dir for every already-computed *_smi_results_dreadd.h5
    file, labeling each by whichever known naming pattern its TSeries
    folder matches. Same disambiguation behavior as discover_animal_sessions
    (Phase 3, Function 3.5) -- nothing is silently dropped on a label
    collision.

    Parameters
    ----------
    animal_dir : str

    Returns
    -------
    catalog : dict
        {label: {'save_path': str, 'session_type': str, 'tseries_dir': str}}
    """
    save_paths = sorted(glob.glob(os.path.join(animal_dir, '**', '*_smi_results_dreadd.h5'),
                                   recursive=True))

    entries = []  # (base_label, session_type, save_path, tseries_dir, tseries_name)
    unmatched = []

    for save_path in save_paths:
        tseries_dir = os.path.dirname(save_path)
        tseries_name = os.path.basename(tseries_dir)
        parent_dir = os.path.dirname(tseries_dir)
        parent_name = os.path.basename(parent_dir)

        upper_tseries = tseries_name.upper()

        if 'SAL' in upper_tseries:
            session_type = 'saline'
            base_label = f'{parent_name}_SALINE'
        elif 'DCZ' in upper_tseries:
            session_type = 'dcz'
            base_label = f'{parent_name}_DCZ'
        else:
            day_match = re.search(r'Day(\d+)', parent_name, re.IGNORECASE)
            if day_match:
                session_type = 'baseline'
                base_label = f'Day{day_match.group(1)}'
            else:
                session_type = 'unknown'
                base_label = tseries_name
                unmatched.append(base_label)

        entries.append((base_label, session_type, save_path, tseries_dir, tseries_name))

    from collections import Counter
    label_counts = Counter(e[0] for e in entries)

    catalog = {}
    for base_label, session_type, save_path, tseries_dir, tseries_name in entries:
        label = f'{base_label}__{tseries_name}' if label_counts[base_label] > 1 else base_label

        if label in catalog:
            print(f"WARNING: label '{label}' still collides after disambiguation -- "
                  f"keeping {catalog[label]['save_path']}, skipping {save_path}")
            continue

        catalog[label] = {
            'save_path': save_path,
            'session_type': session_type,
            'tseries_dir': tseries_dir,
        }

    print(f"Discovered {len(catalog)} sessions with saved SMI results under {animal_dir}:")
    for label, info in catalog.items():
        print(f"  [{info['session_type']:>8}] {label}  <-  {info['save_path']}")

    collided_labels = [l for l, c in label_counts.items() if c > 1]
    if collided_labels:
        print(f"\n{len(collided_labels)} label(s) had multiple TSeries and were disambiguated: "
              f"{collided_labels}")

    if unmatched:
        print(f"\n{len(unmatched)} session(s) didn't match a known naming pattern "
              f"(labeled 'unknown'): {unmatched}")

    return catalog


def select_sessions_popup(session_catalog,
                           title='Select sessions to include in this comparison'):
    """
    Checkbox popup listing every session in session_catalog. Identical to
    Phase 3's function of the same name -- only reads session_type, so it
    works unchanged regardless of whether entries carry a plane0_path or a
    save_path.

    Parameters
    ----------
    session_catalog : dict
    title : str

    Returns
    -------
    selected_labels : list of str
    """
    labels = list(session_catalog.keys())
    n = len(labels)
    display_labels = [f"[{session_catalog[l]['session_type']:>8}] {l}" for l in labels]

    fig_height = max(4, 0.35 * n + 1.5)
    fig = plt.figure(figsize=(9, fig_height))
    try:
        fig.canvas.manager.set_window_title('SELECT SESSIONS -- check boxes, then Confirm (or Enter)')
    except Exception:
        pass

    fig.suptitle(title, fontsize=12, fontweight='bold')

    check_ax = fig.add_axes([0.05, 0.12, 0.9, 0.80])
    check = CheckButtons(check_ax, display_labels, [False] * n)

    confirm_ax = fig.add_axes([0.35, 0.02, 0.3, 0.06])
    confirm_button = Button(confirm_ax, 'Confirm selection')

    state = {'done': False, 'status': [False] * n}

    def on_confirm(event=None):
        state['status'] = list(check.get_status())
        state['done'] = True
        fig.canvas.stop_event_loop()

    confirm_button.on_clicked(on_confirm)

    def on_key(event):
        if event.key == 'enter':
            on_confirm()

    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.show(block=False)
    while not state['done'] and plt.fignum_exists(fig.number):
        fig.canvas.start_event_loop(0.1)

    if not state['done']:
        state['status'] = list(check.get_status())
        print("Window closed without pressing Confirm -- using current checkbox state anyway.")

    if plt.fignum_exists(fig.number):
        plt.close(fig)
        plt.pause(0.01)

    selected_labels = [label for label, checked in zip(labels, state['status']) if checked]
    print(f"Selected {len(selected_labels)} sessions: {selected_labels}")

    return selected_labels


In [5]:
# --- Superseded by Function 4.4's collect_comparison_groups_interactively,
# --- which calls select_sessions_popup itself in a loop. Left commented so
# --- "Run All" doesn't stop here with a redundant one-off picker.
# smi_catalog = discover_smi_sessions(TEST_ANIMAL_DIR)
# selected_smi_labels = select_sessions_popup(smi_catalog)
print("Function 4.2 test cell -- skipped (superseded by Function 4.4 below).")


Function 4.2 test cell -- skipped (superseded by Function 4.4 below).


## Function 4.3 — `build_comparison_table`

Loads every checked session via Function 4.1 and concatenates into one
tidy long-format DataFrame. `condition` is pulled straight from the
catalog's `session_type` (`'baseline'`/`'saline'`/`'dcz'`) — no manual
mapping needed, since which specific sessions you check in the picker is
exactly what defines a given named comparison (e.g. check just Day5 +
DCZ_1's DCZ session for `SalineDCZ_1`; check all of Day1–5 + all three DCZ
sessions for `Baseline_vs_DCZ`).

- **Input:** `session_catalog` (from `discover_smi_sessions`),
  `selected_labels` (from `select_sessions_popup`).
- **Output:** `df` — concatenation of Function 4.1's per-session output
  across all selected sessions, with `session_label`/`condition` columns
  identifying which row came from where.

In [6]:
def build_comparison_table(session_catalog, selected_labels):
    """
    Load every selected session via load_session_smi_for_comparison and
    concatenate into one tidy long-format DataFrame. See markdown above.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.
    selected_labels : list of str
        From select_sessions_popup (or any list of labels you already have).

    Returns
    -------
    df : pandas.DataFrame
    """
    if len(selected_labels) == 0:
        raise ValueError("selected_labels is empty -- nothing to build a comparison table from.")

    session_dfs = []
    for label in selected_labels:
        info = session_catalog[label]
        session_df = load_session_smi_for_comparison(
            info['save_path'], condition_label=info['session_type'], session_label=label
        )
        session_dfs.append(session_df)

    df = pd.concat(session_dfs, ignore_index=True)

    print(f"\nCombined comparison table: {len(df)} cell-rows across {len(selected_labels)} sessions")
    print(df.groupby('condition')['analysis_reliable'].agg(['sum', 'count']))

    return df


In [7]:
# --- Superseded by Function 4.4's run_all_comparisons, which calls
# --- build_comparison_table itself for every defined group. Left commented
# --- so "Run All" doesn't stop here needing the (now-skipped) cell above.
# comparison_df = build_comparison_table(smi_catalog, selected_smi_labels)
# comparison_df.head(10)
print("Function 4.3 test cell -- skipped (superseded by Function 4.4 below).")


Function 4.3 test cell -- skipped (superseded by Function 4.4 below).


## Function 4.4 — manual multi-group loop: name → pick sessions → "more?" → repeat

Ties Functions 4.2/4.3 into the exact workflow you asked for: type a name
for a comparison group, pick its sessions from the checkbox popup, then a
Yes/No popup asks whether to define another group. Answering "Yes" loops
back to naming the next group; "No" closes the loop and moves on to
running every group you defined.

Built as three pieces:
1. **`ask_more_groups_popup`** — small Yes/No button popup, same widget
   style as the checkbox picker. Closing the window without choosing is
   treated as "No, done" rather than hanging forever.
2. **`collect_comparison_groups_interactively`** — the loop itself: prompts
   for a group name (typed at the notebook cell, via `input()`), opens
   `select_sessions_popup` titled with that name, stores
   `{group_name: selected_labels}`, then asks `ask_more_groups_popup` and
   either loops or stops.
3. **`run_all_comparisons`** — once you're done defining groups, loops
   `build_comparison_table` over every one and returns `{group_name: df}`.

- **Input:** `session_catalog` (from `discover_smi_sessions`).
- **Output:** `comparison_groups` (`{group_name: [selected_labels]}`) from
  step 2; `all_group_dfs` (`{group_name: df}`) from step 3.

In [8]:
def ask_more_groups_popup():
    """
    Yes/No popup: "Add another comparison group?"

    Returns
    -------
    bool
        True to define another group, False to stop. Closing the window
        without clicking either button counts as False (stop), so the
        loop can never hang waiting on a window that's gone.
    """
    fig, ax = plt.subplots(figsize=(6, 2.5))
    ax.axis('off')
    fig.suptitle('Add another comparison group?', fontsize=14, fontweight='bold')
    try:
        fig.canvas.manager.set_window_title('MORE GROUPS?')
    except Exception:
        pass

    yes_ax = fig.add_axes([0.12, 0.15, 0.35, 0.35])
    no_ax = fig.add_axes([0.53, 0.15, 0.35, 0.35])
    yes_button = Button(yes_ax, 'Yes, add another')
    no_button = Button(no_ax, 'No, done')

    state = {'choice': None}

    def on_yes(event=None):
        state['choice'] = True
        fig.canvas.stop_event_loop()

    def on_no(event=None):
        state['choice'] = False
        fig.canvas.stop_event_loop()

    yes_button.on_clicked(on_yes)
    no_button.on_clicked(on_no)

    plt.show(block=False)
    while state['choice'] is None and plt.fignum_exists(fig.number):
        fig.canvas.start_event_loop(0.1)

    if plt.fignum_exists(fig.number):
        plt.close(fig)
        plt.pause(0.01)

    if state['choice'] is None:
        print("Window closed without choosing -- treating as 'No, done'.")
        return False

    return state['choice']


def collect_comparison_groups_interactively(session_catalog):
    """
    Loop: type a group name -> pick its sessions -> "more groups?" popup ->
    repeat or stop. See markdown above.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.

    Returns
    -------
    comparison_groups : dict
        {group_name: selected_labels}, in the order you defined them.
    """
    comparison_groups = {}

    while True:
        group_name = input("Name for this comparison group (e.g. 'SalineDCZ_1'): ").strip()
        if not group_name:
            print("Empty name -- skipping this group.")
        elif group_name in comparison_groups:
            print(f"'{group_name}' was already defined -- skipping (pick a different name to redo it).")
        else:
            selected_labels = select_sessions_popup(
                session_catalog, title=f"Select sessions for comparison group '{group_name}'"
            )
            if len(selected_labels) == 0:
                print(f"No sessions selected for '{group_name}' -- not adding this group.")
            else:
                comparison_groups[group_name] = selected_labels
                print(f"Added group '{group_name}': {selected_labels}")

        if not ask_more_groups_popup():
            break

    print(f"\nDefined {len(comparison_groups)} comparison group(s): {list(comparison_groups.keys())}")
    return comparison_groups


def run_all_comparisons(session_catalog, comparison_groups):
    """
    Loop build_comparison_table over every manually-defined group.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.
    comparison_groups : dict
        From collect_comparison_groups_interactively (or any
        {group_name: [labels]} dict you already have).

    Returns
    -------
    all_group_dfs : dict
        {group_name: df} -- df is build_comparison_table's output for
        that group.
    """
    all_group_dfs = {}

    for group_name, selected_labels in comparison_groups.items():
        print(f"\n{'='*90}\nComparison group: {group_name}\n{'='*90}")
        all_group_dfs[group_name] = build_comparison_table(session_catalog, selected_labels)

    print(f"\n{'='*90}\nAll {len(all_group_dfs)} comparison group(s) built: {list(all_group_dfs.keys())}")

    return all_group_dfs


In [9]:
# --- Run through all your comparison groups, one at a time ---
smi_catalog = discover_smi_sessions(TEST_ANIMAL_DIR)

comparison_groups = collect_comparison_groups_interactively(smi_catalog)
all_group_dfs = run_all_comparisons(smi_catalog, comparison_groups)


Discovered 15 sessions with saved SMI results under D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD:
  [baseline] Day1  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260719_JSY_JSY093_LongitudinalImaging_DREADD_Day1\TSeries-07192026-0941-001\TSeries-07192026-0941-001_smi_results_dreadd.h5
  [baseline] Day2  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260720_JSY_JSY093_LongitudinalImaging_DREADD_Day2\TSeries-07202026-1009-001\TSeries-07202026-1009-001_smi_results_dreadd.h5
  [baseline] Day3  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260721_JSY_JSY093_LongitudinalImaging_DREADD_Day3\TSeries-07212026-0907-001\TSeries-07212026-0907-001_smi_results_dreadd.h5
  [baseline] Day4  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260722_JSY_JSY093_LongitudinalImaging_DREADD_Day4\TSeries-07222026-0959-001\TSeries-07222026-0959-001_smi_results_dreadd.h5
  [baseline] Day5  <-  D:\V1_Spati

## Function 4.5 — reliability-fraction vs. trial-count calibration check

Cheap diagnostic, no new expensive computation: for every session that
already has saved SMI results, plot `analysis_reliable` fraction against
that session's trial count and fit a simple trend. If saline's low
reliable fraction sits right on the trend, the gap is plausibly explained
by trial count alone; if it sits noticeably *below* the trend even after
accounting for its lower N, that's a hint of something beyond pure noise.
This only uses data already on disk (no reruns), and directly informs
whether investing in full trial-matched recomputation is worth it before
building it.

**Note for future Track A work** (read this again before building Track
A's version of any saline/DCZ comparison): Track B has no cross-session
cell identity, so there's no way to restrict a comparison to "cells
reliable in both sessions" using each session's own full data — every
Track B comparison necessarily re-establishes "reliable" independently
per session, at that session's own trial count. That means BOTH the
reliable-cell fraction *and* the SMI magnitude among survivors can be
biased by a trial-count difference between conditions (shorter/noisier
sessions can show survivorship bias toward only the most robustly-tuned
cells). Track A's paired within-cell comparison sidesteps this entirely —
the same tracked cell's SMI is compared across sessions regardless of
either session's trial count, so it doesn't need trial-matching to be
valid. This is a real reason to treat Track A as the more decisive test of
the saline-vs-DCZ question once it's built, not just a "more sensitive"
alternative to Track B.

- **`get_session_trial_count`** — tiny helper, reads a session's trial
  count straight from its `preproc.h5` (`spatial_activity.shape[1]`) without
  loading the full array.
- **`build_reliability_trial_count_calibration`** — builds one row per
  session (`n_trials`, `n_reliable`, `reliable_fraction`), fits a simple
  linear trend, and computes each session's residual from that trend.
- **`plot_reliability_calibration`** — scatter of `n_trials` vs.
  `reliable_fraction`, colored by `session_type`, with the fitted trend
  line overlaid.

- **Input:** `session_catalog` (from `discover_smi_sessions`).
- **Output:** `calib_df` (one row per session, with `residual`); the fit
  coefficients; a figure.

In [10]:
def get_session_trial_count(tseries_dir):
    """
    Read a session's trial count straight from its preproc.h5, without
    loading the full spatial_activity array.

    Parameters
    ----------
    tseries_dir : str
        A session's TSeries folder (contains *preproc*.h5).

    Returns
    -------
    n_trials : int
    """
    preproc_files = glob.glob(os.path.join(tseries_dir, "*preproc*.h5"))
    if not preproc_files:
        raise FileNotFoundError(f"No *preproc*.h5 found in {tseries_dir}")
    with h5py.File(preproc_files[0], 'r') as f:
        n_trials = f['spatial_activity'].shape[1]
    return n_trials


def build_reliability_trial_count_calibration(session_catalog):
    """
    For every session with saved SMI results, get (n_trials, reliable_fraction)
    and fit a simple linear trend -- a cheap way to check whether a
    session's reliable-cell fraction is "expected" given how many trials it
    had, or unusually low/high relative to other sessions of similar N.
    See markdown above.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.

    Returns
    -------
    calib_df : pandas.DataFrame
        One row per session: session_label, session_type, n_trials,
        n_cells, n_reliable, reliable_fraction, predicted_fraction,
        residual.
    coeffs : numpy.ndarray
        [slope, intercept] of the fitted line (reliable_fraction ~ n_trials).
    """
    rows = []
    skipped = []
    for label, info in session_catalog.items():
        try:
            n_trials = get_session_trial_count(info['tseries_dir'])
        except FileNotFoundError:
            skipped.append(label)
            continue

        session_df = load_session_smi_for_comparison(info['save_path'], session_label=label)
        n_cells = len(session_df)
        n_reliable = int(session_df['analysis_reliable'].sum())
        rows.append({
            'session_label': label,
            'session_type': info['session_type'],
            'n_trials': n_trials,
            'n_cells': n_cells,
            'n_reliable': n_reliable,
            'reliable_fraction': n_reliable / n_cells if n_cells > 0 else np.nan,
        })

    if skipped:
        print(f"WARNING: {len(skipped)} session(s) skipped -- no *preproc*.h5 found on disk "
              f"(needed for trial count, separate from the already-saved SMI results): {skipped}")

    calib_df = pd.DataFrame(rows)

    coeffs = np.polyfit(calib_df['n_trials'], calib_df['reliable_fraction'], deg=1)
    calib_df['predicted_fraction'] = np.polyval(coeffs, calib_df['n_trials'])
    calib_df['residual'] = calib_df['reliable_fraction'] - calib_df['predicted_fraction']

    print(f"Fitted trend: reliable_fraction ~ {coeffs[0]:.6f} * n_trials + {coeffs[1]:.4f}")
    print("\nSorted by residual (most below-trend first -- worth a closer look):")
    print(calib_df.sort_values('residual')[
        ['session_label', 'session_type', 'n_trials', 'reliable_fraction', 'residual']
    ].to_string(index=False))

    return calib_df, coeffs


def plot_reliability_calibration(calib_df, coeffs):
    """
    Scatter of n_trials vs. reliable_fraction, colored by session_type,
    with the fitted trend line overlaid.

    Parameters
    ----------
    calib_df : pandas.DataFrame
        From build_reliability_trial_count_calibration.
    coeffs : numpy.ndarray
        From build_reliability_trial_count_calibration.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    color_by_type = {'baseline': 'tab:blue', 'saline': 'tab:orange', 'dcz': 'tab:green'}

    fig, ax = plt.subplots(figsize=(9, 8))
    for session_type, group in calib_df.groupby('session_type'):
        ax.scatter(group['n_trials'], group['reliable_fraction'],
                   c=color_by_type.get(session_type, 'gray'), s=80, alpha=0.8,
                   label=session_type)

    x_line = np.linspace(calib_df['n_trials'].min(), calib_df['n_trials'].max(), 100)
    ax.plot(x_line, np.polyval(coeffs, x_line), color='black', linestyle='--',
            label='linear fit')

    ax.set_xlabel('Trial count')
    ax.set_ylabel('Reliable fraction (analysis_reliable)')
    ax.set_title('Reliable-cell fraction vs. trial count')
    ax.legend(loc='best')

    plt.tight_layout()
    return fig


In [11]:
# --- Run the calibration check on every discovered session ---
calib_df, calib_coeffs = build_reliability_trial_count_calibration(smi_catalog)
fig = plot_reliability_calibration(calib_df, calib_coeffs)
plt.show()


Loaded 1048 cells from D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260719_JSY_JSY093_LongitudinalImaging_DREADD_Day1\TSeries-07192026-0941-001\TSeries-07192026-0941-001_smi_results_dreadd.h5
  analysis_reliable: 280, valid: 280
Loaded 1056 cells from D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260720_JSY_JSY093_LongitudinalImaging_DREADD_Day2\TSeries-07202026-1009-001\TSeries-07202026-1009-001_smi_results_dreadd.h5
  analysis_reliable: 150, valid: 150
Loaded 1107 cells from D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260721_JSY_JSY093_LongitudinalImaging_DREADD_Day3\TSeries-07212026-0907-001\TSeries-07212026-0907-001_smi_results_dreadd.h5
  analysis_reliable: 135, valid: 135
Loaded 1057 cells from D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260722_JSY_JSY093_LongitudinalImaging_DREADD_Day4\TSeries-07222026-0959-001\TSeries-07222026-0959-001_smi_results_dreadd.h5
  analysis_reliable: 144, valid: 144
Load

## Function 4.6 — decompose reliability into its components, test each for a condition effect beyond trial count

`combined_reliable = reliable_cells & pattern_reliable & active_cells`.
Function 3.3b found that of cells failing `combined_reliable`,
`pattern_unstable` (peak-location stability) accounted for 94.6% of
failures, while `reliable_cells`'s own flat floors (`avg_cc`/`cohen_d`)
each accounted for ~21%, and the shuffle-significance step (the one part
of `reliable_cells` that isn't recomputable without a full rerun) excluded
**zero** additional cells beyond those floors. So lumping everything into
`combined_reliable` can't tell you *which* aspect of reliability a
trial-count-independent condition effect is coming from — `reliable_cells`
(≈ is activity level reproducible trial-to-trial) and `pattern_reliable`
(≈ is peak *location* reproducible) mean different things, and the
question raised above (does higher reliability under DCZ mean anything
about spatial coding specifically) depends on which one is driving it.

Both are cheap to get without any expensive shuffle rerun:
`reliable_cells` uses a proxy (`avg_cc > threshold` & `cohen_d > threshold`,
already saved, free) — justified by the zero-additional-exclusions finding
above. `pattern_reliable`/`active_cells` need `spatial_activity` reloaded
from `preproc.h5` (cheap I/O) run through the same imported, unmodified
functions Function 3.3b already used — no shuffling.

Built as four pieces:
1. **`compute_reliability_components_for_session`** — per-session
   `reliable_cells`/`pattern_reliable`/`active` fractions.
2. **`build_reliability_component_calibration`** — one table with all four
   fractions (`combined_reliable` plus its three components) per session,
   alongside `n_trials`/`session_type` (parallel to, but a distinct table
   from, Function 4.5's `analysis_reliable`-based one — this one uses the
   *true* `combined_reliable`, not `analysis_reliable`, so the three
   components multiply together to reconstruct it exactly).
3. **`test_condition_effect_on_reliability`** — weighted least squares
   (`fraction ~ n_trials + C(session_type)`, weighted by `n_cells`),
   `baseline` as the reference level, plus a direct DCZ-vs-saline contrast.
4. **`test_all_reliability_components`** — runs (3) for all four fraction
   columns, side by side.

**Caveat**: 15 sessions, one animal, 3 model parameters — treat p-values as
a pilot signal, not a conclusion.

- **Input:** `session_catalog`.
- **Output:** `component_calib_df`; `{fraction_col: model_result}`.

In [12]:
def compute_reliability_components_for_session(tseries_dir, avg_cc, cohen_d,
                                                min_cc_threshold=0.1, cohen_threshold=0.8,
                                                min_pattern_corr=0.3, peak_distance_threshold=5,
                                                activity_method='absolute_percentile'):
    """
    Per-session reliable_cells/pattern_reliable/active fractions. See
    markdown above for why reliable_cells uses a proxy rather than a full
    rerun, and why pattern_reliable/active_cells need spatial_activity
    reloaded but not shuffled.

    Parameters
    ----------
    tseries_dir : str
        A session's TSeries folder (contains *preproc*.h5).
    avg_cc, cohen_d : numpy.ndarray
        Already-saved per-cell values (e.g. from load_session_smi_for_comparison).
    min_cc_threshold, cohen_threshold, min_pattern_corr, peak_distance_threshold : float
        Must match Preprocess.py's actual call for this session's area
        (same caveat as Function 3.3b).
    activity_method : str

    Returns
    -------
    dict with 'reliable_cells_fraction', 'pattern_reliable_fraction', 'active_fraction'.
    """
    preproc_files = glob.glob(os.path.join(tseries_dir, "*preproc*.h5"))
    if not preproc_files:
        raise FileNotFoundError(f"No *preproc*.h5 found in {tseries_dir}")
    preproc_data = files.read_h5(preproc_files[0])
    spatial_activity = preproc_data['spatial_activity']

    n_cells = spatial_activity.shape[0]

    active_cells, _ = improved_activity_threshold_check(spatial_activity, method=activity_method)
    pattern_reliable, odd_even_corr, peak_distances = evaluate_pattern_similarity_improved(
        spatial_activity, min_pattern_corr, peak_distance_threshold
    )

    # Proxy for reliable_cells -- flat floors only, skipping the per-cell
    # shuffle-significance test (expensive; Function 3.3b found it excluded
    # zero additional cells beyond these floors on the Day1 test session).
    reliable_cells_proxy = (avg_cc > min_cc_threshold) & (cohen_d > cohen_threshold)

    return {
        'reliable_cells_fraction': float(np.sum(reliable_cells_proxy)) / n_cells,
        'pattern_reliable_fraction': float(np.sum(pattern_reliable)) / n_cells,
        'active_fraction': float(np.sum(active_cells)) / n_cells,
    }


def build_reliability_component_calibration(session_catalog,
                                             min_cc_threshold=0.1, cohen_threshold=0.8,
                                             min_pattern_corr=0.3, peak_distance_threshold=5,
                                             activity_method='absolute_percentile'):
    """
    One table with combined_reliable plus its three components
    (reliable_cells proxy, pattern_reliable, active) per session, alongside
    n_trials/session_type/n_cells. See markdown above.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.
    min_cc_threshold, cohen_threshold, min_pattern_corr, peak_distance_threshold : float
    activity_method : str

    Returns
    -------
    calib_df : pandas.DataFrame
    """
    rows = []
    skipped = []
    for label, info in session_catalog.items():
        try:
            n_trials = get_session_trial_count(info['tseries_dir'])
        except FileNotFoundError:
            skipped.append(label)
            continue

        session_df = load_session_smi_for_comparison(info['save_path'], session_label=label)
        n_cells = len(session_df)
        avg_cc = session_df['avg_cc'].to_numpy()
        cohen_d = session_df['cohen_d'].to_numpy()
        combined_reliable = session_df['combined_reliable'].to_numpy()

        components = compute_reliability_components_for_session(
            info['tseries_dir'], avg_cc, cohen_d,
            min_cc_threshold=min_cc_threshold, cohen_threshold=cohen_threshold,
            min_pattern_corr=min_pattern_corr, peak_distance_threshold=peak_distance_threshold,
            activity_method=activity_method,
        )

        rows.append({
            'session_label': label,
            'session_type': info['session_type'],
            'n_trials': n_trials,
            'n_cells': n_cells,
            'combined_reliable_fraction': float(np.sum(combined_reliable)) / n_cells,
            'reliable_cells_fraction': components['reliable_cells_fraction'],
            'pattern_reliable_fraction': components['pattern_reliable_fraction'],
            'active_fraction': components['active_fraction'],
        })

    if skipped:
        print(f"WARNING: {len(skipped)} session(s) skipped -- no *preproc*.h5 found: {skipped}")

    calib_df = pd.DataFrame(rows)

    print(f"Built component calibration table for {len(calib_df)} sessions:\n")
    print(calib_df[['session_label', 'session_type', 'n_trials',
                     'combined_reliable_fraction', 'reliable_cells_fraction',
                     'pattern_reliable_fraction', 'active_fraction']].to_string(index=False))

    return calib_df


def test_condition_effect_on_reliability(calib_df, fraction_col, weight_col='n_cells',
                                          reference_level='baseline'):
    """
    Weighted least squares: fraction_col ~ n_trials + C(session_type),
    weighted by n_cells (sessions with more cells get more weight, since
    their fraction estimate is less noisy). Tests whether session_type
    predicts fraction_col beyond what n_trials alone explains.

    Parameters
    ----------
    calib_df : pandas.DataFrame
        From build_reliability_component_calibration (or any table with
        fraction_col, n_trials, session_type, weight_col).
    fraction_col : str
        Which fraction column to test.
    weight_col : str
    reference_level : str
        Which session_type is the reference level for the categorical
        dummy coding (baseline vs. saline/dcz coefficients are relative
        to this).

    Returns
    -------
    model_result : statsmodels regression results object
    """
    df = calib_df.copy()
    other_levels = [c for c in df['session_type'].unique() if c != reference_level]
    df['session_type'] = pd.Categorical(df['session_type'], categories=[reference_level] + other_levels)

    formula = f"{fraction_col} ~ n_trials + C(session_type)"
    model_result = smf.wls(formula, data=df, weights=df[weight_col]).fit()

    print(f"\n=== {fraction_col} ~ n_trials + condition (weighted by {weight_col}, "
          f"reference='{reference_level}') ===")
    print(model_result.summary().tables[1])

    param_names = list(model_result.params.index)
    dcz_name = next((p for p in param_names if 'dcz' in p.lower()), None)
    saline_name = next((p for p in param_names if 'saline' in p.lower()), None)
    if dcz_name and saline_name:
        contrast = f"{dcz_name} - {saline_name}"
        print(f"\nDirect DCZ vs. saline contrast:")
        print(model_result.t_test(contrast))

    return model_result


def test_all_reliability_components(calib_df, weight_col='n_cells', reference_level='baseline'):
    """
    Runs test_condition_effect_on_reliability for all four fraction
    columns, side by side.

    Parameters
    ----------
    calib_df : pandas.DataFrame
        From build_reliability_component_calibration.
    weight_col : str
    reference_level : str

    Returns
    -------
    results : dict
        {fraction_col: model_result}.
    """
    fraction_cols = ['combined_reliable_fraction', 'reliable_cells_fraction',
                      'pattern_reliable_fraction', 'active_fraction']
    results = {}
    for col in fraction_cols:
        results[col] = test_condition_effect_on_reliability(
            calib_df, col, weight_col=weight_col, reference_level=reference_level
        )
    return results


In [13]:
# --- Build the component table and test each component for a condition effect ---
component_calib_df = build_reliability_component_calibration(smi_catalog)
component_results = test_all_reliability_components(component_calib_df)


Loaded 1048 cells from D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260719_JSY_JSY093_LongitudinalImaging_DREADD_Day1\TSeries-07192026-0941-001\TSeries-07192026-0941-001_smi_results_dreadd.h5
  analysis_reliable: 280, valid: 280
Activity threshold (absolute_percentile): 0.0920
Cells passing activity threshold: 943/1048 (90.0%)

Pattern Similarity Results:
  Mean odd-even correlation: 0.432
  Mean peak distance: 18.4 bins
  Cells with good correlation (>0.3): 727
  Cells with stable peaks (<5 bins): 599
Loaded 1056 cells from D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260720_JSY_JSY093_LongitudinalImaging_DREADD_Day2\TSeries-07202026-1009-001\TSeries-07202026-1009-001_smi_results_dreadd.h5
  analysis_reliable: 150, valid: 150
Activity threshold (absolute_percentile): 0.1358
Cells passing activity threshold: 950/1056 (90.0%)

Pattern Similarity Results:
  Mean odd-even correlation: 0.312
  Mean peak distance: 28.1 bins
  Cells with good correlati

## Function 4.7 — compare SMI distributions across conditions (the actual hypothesis test)

Everything through Function 4.6 was about whether *reliability* differs by
condition — a necessary detour (a badly-behaved reliability comparison
would corrupt any SMI comparison built on top of it), but not itself a
test of spatial coding. This is that test.

Filters to `valid` cells — Function 4.1's `valid` column is already
`reliable_valid_cells` (reliable **and** the SMI curve fit succeeded), the
exact mask `run_smi_analysis_session` itself uses for layer-level SMI
stats, so no extra AND-ing is needed.

Built as three pieces:
1. **`compare_smi_across_conditions`** — Kruskal-Wallis omnibus across
   whatever conditions are present in a group, then pairwise Mann-Whitney U
   for every condition pair, Holm-corrected within that group. Returns
   `None` (with a message) for single-condition groups like `baseline`,
   rather than erroring.
2. **`plot_smi_comparison`** — violin + strip plot of SMI by condition,
   filtered to `valid` cells, standard rcParams.
3. **`run_all_group_comparisons`** — loops both over every group in
   `all_group_dfs`, skipping single-condition groups automatically.

Non-parametric (Kruskal-Wallis / Mann-Whitney) rather than ANOVA/t-test,
since SMI is bounded and likely non-normal, and this doesn't assume equal
variance between conditions.

- **Input:** `all_group_dfs` (from Function 4.4) or any single group's `df`.
- **Output:** per-group omnibus + pairwise stats table; a figure per group.

In [14]:
def compare_smi_across_conditions(df, group_col='condition', value_col='SMI', filter_col='valid'):
    """
    Kruskal-Wallis omnibus + pairwise Mann-Whitney U (Holm-corrected) across
    whatever conditions are present. See markdown above.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table (e.g. from build_comparison_table).
    group_col : str
    value_col : str
    filter_col : str
        Boolean column to filter to before comparing (default 'valid' ==
        reliable_valid_cells).

    Returns
    -------
    result : dict or None
        None (with a printed message) if fewer than 2 conditions remain
        after filtering. Otherwise:
        {'group_medians': {cond: median}, 'group_n': {cond: n},
         'omnibus_stat', 'omnibus_p',
         'pairwise': DataFrame(cond_a, cond_b, median_diff, U_stat, p_raw, p_holm)}.
    """
    filtered = df[df[filter_col]]
    conditions = [c for c in filtered[group_col].unique() if pd.notna(c)]

    if len(conditions) < 2:
        print(f"Only {len(conditions)} condition(s) present after filtering on '{filter_col}' "
              f"-- nothing to compare ({conditions}).")
        return None

    samples = {cond: filtered.loc[filtered[group_col] == cond, value_col].to_numpy()
               for cond in conditions}

    group_medians = {cond: float(np.median(vals)) for cond, vals in samples.items()}
    group_n = {cond: len(vals) for cond, vals in samples.items()}

    omnibus_stat, omnibus_p = kruskal(*samples.values())

    pairwise_rows = []
    for cond_a, cond_b in combinations(conditions, 2):
        u_stat, p_raw = mannwhitneyu(samples[cond_a], samples[cond_b], alternative='two-sided')
        pairwise_rows.append({
            'cond_a': cond_a, 'cond_b': cond_b,
            'median_diff': group_medians[cond_a] - group_medians[cond_b],
            'U_stat': u_stat, 'p_raw': p_raw,
        })

    pairwise_df = pd.DataFrame(pairwise_rows)
    if len(pairwise_df) > 0:
        _, p_holm, _, _ = multipletests(pairwise_df['p_raw'], method='holm')
        pairwise_df['p_holm'] = p_holm

    print(f"Conditions: {conditions}")
    print(f"  n per condition: {group_n}")
    print(f"  median {value_col} per condition: {group_medians}")
    print(f"  Kruskal-Wallis: H={omnibus_stat:.3f}, p={omnibus_p:.4f}")
    print(f"\n  Pairwise (Holm-corrected):")
    print(pairwise_df.to_string(index=False))

    return {
        'group_medians': group_medians,
        'group_n': group_n,
        'omnibus_stat': omnibus_stat,
        'omnibus_p': omnibus_p,
        'pairwise': pairwise_df,
    }


def _order_categories(values):
    """
    Order category values for plotting/display: condition-style values
    ('baseline'/'saline'/'dcz') get that fixed order; 'DayN'-style values
    (e.g. the baseline group's session_label fallback) get sorted
    numerically by day; anything else keeps first-seen order.

    Parameters
    ----------
    values : array-like

    Returns
    -------
    list
    """
    unique_vals = list(pd.unique(values))
    condition_order = [c for c in ['baseline', 'saline', 'dcz'] if c in unique_vals]
    if len(condition_order) == len(unique_vals):
        return condition_order

    day_pattern = re.compile(r'Day(\d+)', re.IGNORECASE)
    if len(unique_vals) > 0 and all(day_pattern.fullmatch(str(v)) for v in unique_vals):
        return sorted(unique_vals, key=lambda v: int(day_pattern.fullmatch(str(v)).group(1)))

    remaining = [v for v in unique_vals if v not in condition_order]
    return condition_order + remaining


def plot_smi_comparison(df, group_col='condition', value_col='SMI', filter_col='valid', title=''):
    """
    Violin + strip plot of SMI by group_col, filtered to valid cells.

    Parameters
    ----------
    df : pandas.DataFrame
    group_col, value_col, filter_col : str
    title : str

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    filtered = df[df[filter_col]]
    condition_order = _order_categories(filtered[group_col])

    color_by_condition = {'baseline': 'tab:blue', 'saline': 'tab:orange', 'dcz': 'tab:green'}

    fig, ax = plt.subplots(figsize=(8, 8))
    data_by_condition = [filtered.loc[filtered[group_col] == cond, value_col].to_numpy()
                         for cond in condition_order]

    parts = ax.violinplot(data_by_condition, showmedians=True)
    for i, body in enumerate(parts['bodies']):
        body.set_facecolor(color_by_condition.get(condition_order[i], 'gray'))
        body.set_alpha(0.4)

    rng = np.random.default_rng(0)
    for i, vals in enumerate(data_by_condition):
        jitter = rng.uniform(-0.08, 0.08, size=len(vals))
        ax.scatter(np.full(len(vals), i + 1) + jitter, vals,
                   color=color_by_condition.get(condition_order[i], 'gray'),
                   s=15, alpha=0.5)

    ax.set_xticks(range(1, len(condition_order) + 1))
    ax.set_xticklabels(condition_order)
    ax.set_ylabel(value_col)
    ax.set_title(title)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

    plt.tight_layout()
    return fig


def run_all_group_comparisons(all_group_dfs, group_col='condition', value_col='SMI', filter_col='valid',
                               fallback_group_col='session_label'):
    """
    Loops compare_smi_across_conditions + plot_smi_comparison over every
    group. If a group has fewer than 2 distinct values of group_col (e.g.
    the 'baseline' group, where 'condition' is constant), falls back to
    fallback_group_col instead ('session_label' -- Day1..Day5) rather than
    skipping it outright, so you still see whether SMI drifts across the
    baseline days themselves (a direct check on the "maybe it's
    training-day drift, not the drug" alternative explanation raised
    earlier).

    Each returned result dict includes a 'fig' key (the figure just
    plotted) alongside the stats, so Function 4.8 can save it later
    without needing to replot from scratch.

    Parameters
    ----------
    all_group_dfs : dict
        {group_name: df}, from Function 4.4's run_all_comparisons.
    group_col, value_col, filter_col : str
    fallback_group_col : str or None
        Set to None to restore the old skip-only behavior.

    Returns
    -------
    results : dict
        {group_name: compare_smi_across_conditions(...) result, plus a
        'fig' key} -- only for groups where either grouping produced 2+
        categories.
    """
    results = {}
    for group_name, df in all_group_dfs.items():
        print(f"\n{'='*90}\n{group_name}\n{'='*90}")
        used_group_col = group_col
        result = compare_smi_across_conditions(df, group_col=group_col, value_col=value_col,
                                                filter_col=filter_col)

        if result is None and fallback_group_col is not None and fallback_group_col != group_col:
            print(f"Falling back to grouping by '{fallback_group_col}' instead...")
            used_group_col = fallback_group_col
            result = compare_smi_across_conditions(df, group_col=fallback_group_col, value_col=value_col,
                                                    filter_col=filter_col)

        if result is not None:
            fig = plot_smi_comparison(df, group_col=used_group_col, value_col=value_col,
                                      filter_col=filter_col,
                                      title=f"{group_name} (by {used_group_col})")
            result['fig'] = fig
            results[group_name] = result
            plt.show()

    return results


In [15]:
# --- Run the actual hypothesis test across all your defined comparison groups ---
smi_comparison_results = run_all_group_comparisons(all_group_dfs)



dcz1
Conditions: ['dcz', 'saline']
  n per condition: {'dcz': 188, 'saline': 91}
  median SMI per condition: {'dcz': 0.7569123811114304, 'saline': 0.8447094255963993}
  Kruskal-Wallis: H=12.806, p=0.0003

  Pairwise (Holm-corrected):
cond_a cond_b  median_diff  U_stat    p_raw   p_holm
   dcz saline    -0.087797  6293.0 0.000346 0.000346

dcz2
Conditions: ['dcz', 'saline']
  n per condition: {'dcz': 62, 'saline': 98}
  median SMI per condition: {'dcz': 0.4799209447355657, 'saline': 0.6181093891483138}
  Kruskal-Wallis: H=1.643, p=0.1999

  Pairwise (Holm-corrected):
cond_a cond_b  median_diff  U_stat    p_raw   p_holm
   dcz saline    -0.138188  2672.0 0.200497 0.200497

dcz3
Conditions: ['dcz', 'saline']
  n per condition: {'dcz': 107, 'saline': 53}
  median SMI per condition: {'dcz': 0.6352165981814266, 'saline': 0.6827961177321634}
  Kruskal-Wallis: H=1.903, p=0.1678

  Pairwise (Holm-corrected):
cond_a cond_b  median_diff  U_stat    p_raw   p_holm
   dcz saline     -0.04758  2455.

## Diagnostic -- response plots for the actual SMI-comparison population

`compare_smi_across_conditions` (Function 4.7, above) only ever sees cells filtered to `valid` -- `global_smi/valid_cells_mask`, i.e. `analysis_reliable_cells` (passed the even/odd reliability test AND doesn't peak in the onset/reward exclusion zones) further restricted to cells where the SMI curve fit itself succeeded. Before trusting a group-level SMI difference, worth actually looking at that population: how many cells, and what do their response profiles look like -- e.g. is one condition's population dominated by a cluster of cells peaking at the same corridor position, which would shift the group SMI distribution for compositional reasons having nothing to do with a real change in spatial coding.

Reuses `helper/ResponseVisualization.py`'s `create_response_plot` completely unmodified (odd-trial peak sort, each panel independently normalized to its own [0, 1] -- its own existing behavior, not changed here) for three cell-selection masks, saline then dcz each time (6 plots total):

- **`analysis_reliable_cells`** ("reliable") -- passed the reliability test, not onset/reward-peaking.
- **`valid_cells_mask`** ("reliable-valid") -- the above AND the SMI fit succeeded. The exact population Function 4.7 compares.
- **`analysis_reliable_cells & ~valid_cells_mask`** ("reliable, rejected from valid") -- consistent, non-onset/reward cells that still got dropped from the SMI comparison because the curve-fitting procedure itself failed for them. Shows what "reliable" is silently leaving out of "reliable-valid," and whether that differs saline vs. dcz.

In [16]:
from helper.ResponseVisualization import create_response_plot


def plot_response_plot_diagnostics(session_catalog, saline_label, dcz_label):
    """
    For one saline/dcz session pair, run create_response_plot (reused
    unmodified from helper/ResponseVisualization.py -- odd-trial peak
    sort, each panel independently normalized to its own [0, 1]) on three
    cell-selection masks, saline then dcz each time (6 plots total):

    - analysis_reliable_cells ('reliable') -- passed the even/odd
      reliability test and doesn't peak in the onset/reward exclusion
      zones.
    - valid_cells_mask ('reliable-valid') -- the above AND the SMI curve
      fit succeeded. This is the ONLY population
      compare_smi_across_conditions (Function 4.7) actually compares.
    - analysis_reliable_cells & ~valid_cells_mask ('reliable, rejected
      from valid') -- cells that passed reliability but got dropped from
      the SMI comparison anyway, because the curve-fitting procedure
      itself failed for them.

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.
    saline_label, dcz_label : str
        Must be keys in session_catalog.

    Returns
    -------
    figs : dict
        {(mask_name, label): fig} -- 6 entries (3 masks x 2 sessions,
        saline before dcz within each mask).
    sorted_indices : dict
        {(mask_name, label): sorted_reliable_indices} --
        create_response_plot's own second return value, per call.
    """
    def _load_session_masks(label):
        info = session_catalog[label]
        preproc_files = glob.glob(os.path.join(info['tseries_dir'], "*preproc*.h5"))
        if not preproc_files:
            raise FileNotFoundError(f"No *preproc*.h5 found in {info['tseries_dir']} for '{label}'")
        preproc_data = files.read_h5(preproc_files[0])
        norm_spatial_activity = preproc_data['norm_spatial_activity']

        with h5py.File(info['save_path'], 'r') as f:
            analysis_reliable_cells = f['global_smi/analysis_reliable_cells'][:]
            valid_cells_mask = f['global_smi/valid_cells_mask'][:]

        masks = {
            'reliable': analysis_reliable_cells,
            'reliable-valid': valid_cells_mask,
            'reliable, rejected from valid': analysis_reliable_cells & ~valid_cells_mask,
        }
        return norm_spatial_activity, masks

    session_data = {
        saline_label: _load_session_masks(saline_label),
        dcz_label: _load_session_masks(dcz_label),
    }

    print(f"Cell counts -- '{saline_label}' vs '{dcz_label}':")
    for mask_name in ('reliable', 'reliable-valid', 'reliable, rejected from valid'):
        n_saline = int(session_data[saline_label][1][mask_name].sum())
        n_dcz = int(session_data[dcz_label][1][mask_name].sum())
        print(f"  {mask_name:32s}  saline={n_saline:4d}   dcz={n_dcz:4d}")

    mask_order = ['reliable', 'reliable-valid', 'reliable, rejected from valid']
    figs = {}
    sorted_indices = {}
    for mask_name in mask_order:
        for label in (saline_label, dcz_label):  # saline always plotted first
            norm_spatial_activity, masks = session_data[label]
            mask = masks[mask_name]
            fig, sorted_idx = create_response_plot(norm_spatial_activity, mask, clim=(0, 1))
            fig.suptitle(f"{label}" + chr(10) + f"{mask_name} (n={int(mask.sum())})", fontsize=14)
            figs[(mask_name, label)] = fig
            sorted_indices[(mask_name, label)] = sorted_idx
            plt.show()

    return figs, sorted_indices

In [17]:
# --- Try it on the real saline/dcz session pair for whichever animal
# TEST_ANIMAL_DIR (set in the first cell) currently points at -- looked up
# by animal dir rather than hardcoded to one animal, so it stays correct
# whichever of TEST_ANIMAL_DIR_JSY090/_JSY093 is currently "active" above,
# without needing two separate blocks to keep in sync by hand.
RESPPLOT_LABELS_BY_ANIMAL_DIR = {
    TEST_ANIMAL_DIR_JSY090: {
        'saline_label': '260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_SALINE',
        'dcz_label': '260724_JSY_JSY090_LongitudinalImaging_DREADD_Saline_DCZ_1_DCZ',
    },
    TEST_ANIMAL_DIR_JSY093: {
        'saline_label': '260724_JSY_JSY093_LongitudinalImaging_DREADD_Saline_DCZ_1_SALINE',
        'dcz_label': '260724_JSY_JSY093_LongitudinalImaging_DREADD_Saline_DCZ_1_DCZ',
    },
}

if TEST_ANIMAL_DIR not in RESPPLOT_LABELS_BY_ANIMAL_DIR:
    raise ValueError(f"No response-plot test labels defined for TEST_ANIMAL_DIR={TEST_ANIMAL_DIR!r} "
                      f"-- add an entry to RESPPLOT_LABELS_BY_ANIMAL_DIR above.")

respplot_labels = RESPPLOT_LABELS_BY_ANIMAL_DIR[TEST_ANIMAL_DIR]
active_smi_catalog = discover_smi_sessions(TEST_ANIMAL_DIR)  # fresh -- always matches TEST_ANIMAL_DIR exactly

respplot_figs, respplot_sorted_indices = plot_response_plot_diagnostics(
    active_smi_catalog, respplot_labels['saline_label'], respplot_labels['dcz_label']
)

Discovered 15 sessions with saved SMI results under D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD:
  [baseline] Day1  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260719_JSY_JSY093_LongitudinalImaging_DREADD_Day1\TSeries-07192026-0941-001\TSeries-07192026-0941-001_smi_results_dreadd.h5
  [baseline] Day2  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260720_JSY_JSY093_LongitudinalImaging_DREADD_Day2\TSeries-07202026-1009-001\TSeries-07202026-1009-001_smi_results_dreadd.h5
  [baseline] Day3  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260721_JSY_JSY093_LongitudinalImaging_DREADD_Day3\TSeries-07212026-0907-001\TSeries-07212026-0907-001_smi_results_dreadd.h5
  [baseline] Day4  <-  D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\260722_JSY_JSY093_LongitudinalImaging_DREADD_Day4\TSeries-07222026-0959-001\TSeries-07222026-0959-001_smi_results_dreadd.h5
  [baseline] Day5  <-  D:\V1_Spati

### Response-plot diagnostics for whatever groups you actually selected in the popup

The single-pair test above is just a fixed sanity check (DCZ1). This version follows `comparison_groups` -- whatever you picked via Function 4.4's `collect_comparison_groups_interactively` popup -- so the response-plot diagnostic always matches the exact same groups Function 4.7's SMI comparison (`smi_comparison_results = run_all_group_comparisons(all_group_dfs)`, above) is actually testing, instead of a separate hardcoded pair that could silently drift out of sync with what you're really comparing.

A group is skipped (with a printed reason, not an error) if it doesn't contain exactly one saline and one dcz session -- e.g. a baseline-only group (Day1-Day5) has neither, so the diagnostic doesn't apply to it.

In [18]:
def plot_response_plot_diagnostics_for_groups(session_catalog, comparison_groups):
    """
    Loop plot_response_plot_diagnostics over every comparison group you
    defined interactively (Function 4.4's
    collect_comparison_groups_interactively) -- so the response-plot
    diagnostic always matches whatever Function 4.7 is actually comparing,
    instead of a separate hardcoded session pair. A group is skipped
    (with a printed reason) if it doesn't contain exactly one saline and
    one dcz session -- the diagnostic only makes sense for a direct
    saline-vs-dcz pair (e.g. a baseline-only group has neither).

    Parameters
    ----------
    session_catalog : dict
        From discover_smi_sessions.
    comparison_groups : dict
        {group_name: selected_labels}, from
        collect_comparison_groups_interactively (or any {name: [labels]}
        dict you already have).

    Returns
    -------
    results_by_group : dict
        {group_name: (figs, sorted_indices)} -- plot_response_plot_diagnostics's
        own return value, per group -- only for groups that actually ran.
    """
    results_by_group = {}
    for group_name, selected_labels in comparison_groups.items():
        saline_labels = [l for l in selected_labels
                          if session_catalog.get(l, {}).get('session_type') == 'saline']
        dcz_labels = [l for l in selected_labels
                      if session_catalog.get(l, {}).get('session_type') == 'dcz']

        if len(saline_labels) != 1 or len(dcz_labels) != 1:
            print(f"Skipping '{group_name}' -- needs exactly 1 saline + 1 dcz session, found "
                  f"{len(saline_labels)} saline, {len(dcz_labels)} dcz (this diagnostic only "
                  f"applies to a direct saline-vs-dcz pair).")
            continue

        print(f"\n{'='*90}\nResponse-plot diagnostic -- {group_name}\n{'='*90}")
        figs, sorted_indices = plot_response_plot_diagnostics(session_catalog, saline_labels[0], dcz_labels[0])
        results_by_group[group_name] = (figs, sorted_indices)

    return results_by_group

In [19]:
# --- Response-plot diagnostics for the SAME groups the SMI comparison above just ran on ---
respplot_results_by_group = plot_response_plot_diagnostics_for_groups(smi_catalog, comparison_groups)


Response-plot diagnostic -- dcz1
Cell counts -- '260724_JSY_JSY093_LongitudinalImaging_DREADD_Saline_DCZ_1_SALINE' vs '260724_JSY_JSY093_LongitudinalImaging_DREADD_Saline_DCZ_1_DCZ':
  reliable                          saline=  91   dcz= 188
  reliable-valid                    saline=  91   dcz= 188
  reliable, rejected from valid     saline=   0   dcz=   0

Response-plot diagnostic -- dcz2
Cell counts -- '260726_JSY_JSY093_LongitudinalImaging_DREADD_Saline_DCZ_2_SALINE__TSeries-07262026-0755_SAL-001' vs '260726_JSY_JSY093_LongitudinalImaging_DREADD_Saline_DCZ_2_DCZ':
  reliable                          saline=  98   dcz=  62
  reliable-valid                    saline=  98   dcz=  62
  reliable, rejected from valid     saline=   0   dcz=   0

Response-plot diagnostic -- dcz3
Cell counts -- '260728_JSY_JSY093_LongitudinalImaging_DREADD_Saline_DCZ_3_SALINE' vs '260728_JSY_JSY093_LongitudinalImaging_DREADD_Saline_DCZ_3_DCZ':
  reliable                          saline=  53   dcz= 107
  re

## Function 4.8 — save everything Phase 4 generates

Nothing in this phase was persisted to disk until now — `all_group_dfs`,
Function 4.7's stats, and every figure only ever lived in the kernel's
memory. This saves all of it per animal, so future comparisons (Phase 5+,
or just re-inspecting later) don't require re-running the interactive
picker from scratch.

Saved under `{ANIMAL_DIR}/Phase4_SessionComparison_Results/`:
- `{group}_comparison_table.csv` — the raw per-cell table for every group.
- `{group}_smi_pairwise_stats.csv` + `{group}_smi_stats_summary.json` —
  Function 4.7's pairwise Mann-Whitney table + omnibus/medians/n.
- `{group}_smi_comparison.png` — Function 4.7's violin plot per group.
- `reliability_calibration.csv` / `_fit.json` / `_plot.png` — Function
  4.5's outputs, if you ran it.
- `reliability_components.csv` + `reliability_regression_{fraction_col}.txt`
  (one per fraction column) — Function 4.6's outputs, if you ran it.

Built as small single-purpose pieces rather than one large function, so
any of them can be reused independently later (e.g. Phase 5 saving its
own layer-specific figures the same way):
1. **`save_dataframe_csv`** / **`save_figure_png`** / **`save_json`** —
   tiny generic helpers, all creating `output_dir` if it doesn't exist yet.
2. **`save_comparison_group_outputs`** — one group's table + stats +
   figure.
3. **`save_all_comparison_outputs`** — loops (2) over every group.
4. **`save_reliability_calibration_outputs`** / **`save_reliability_component_outputs`**
   — Functions 4.5/4.6's outputs.
5. **`save_all_phase4_outputs`** — top-level convenience wrapping
   everything above in one call.

- **Input:** `output_dir`, `all_group_dfs`, `smi_comparison_results` (now
  carrying a `'fig'` key per group, added by Function 4.7's update above);
  optionally Function 4.5/4.6's outputs too.
- **Output:** `saved_paths` (nested dict of every path written).

In [20]:
import json


def save_dataframe_csv(df, output_dir, filename):
    """
    Save a DataFrame to {output_dir}/{filename}, creating output_dir if
    needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    df.to_csv(save_path, index=False)
    print(f"Saved -> {save_path}")
    return save_path


def save_figure_png(fig, output_dir, filename, dpi=150):
    """
    Save a matplotlib figure to {output_dir}/{filename}, creating
    output_dir if needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    print(f"Saved -> {save_path}")
    return save_path


def _json_safe(obj):
    """Recursively convert numpy scalar types to native Python for json.dump."""
    if isinstance(obj, dict):
        return {k: _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


def save_json(data, output_dir, filename):
    """
    Save a JSON-serializable dict to {output_dir}/{filename}, creating
    output_dir if needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    with open(save_path, 'w') as f:
        json.dump(_json_safe(data), f, indent=2)
    print(f"Saved -> {save_path}")
    return save_path


def save_comparison_group_outputs(output_dir, group_name, df, stats_result=None):
    """
    Save one comparison group's outputs: the raw per-cell table (CSV),
    and -- if stats_result is provided (a compare_smi_across_conditions
    result, with the 'fig' key run_all_group_comparisons now adds) -- the
    pairwise stats table (CSV), a summary of medians/n/omnibus (JSON), and
    the figure (PNG).

    Parameters
    ----------
    output_dir : str
    group_name : str
    df : pandas.DataFrame
        The group's raw per-cell table (from all_group_dfs).
    stats_result : dict or None
        From run_all_group_comparisons's per-group result (includes 'fig').

    Returns
    -------
    saved_paths : dict
        {'table', 'pairwise', 'summary', 'figure'} -> path (only keys
        that were actually saved).
    """
    saved_paths = {'table': save_dataframe_csv(df, output_dir, f"{group_name}_comparison_table.csv")}

    if stats_result is not None:
        saved_paths['pairwise'] = save_dataframe_csv(
            stats_result['pairwise'], output_dir, f"{group_name}_smi_pairwise_stats.csv"
        )
        summary = {
            'group_medians': stats_result['group_medians'],
            'group_n': stats_result['group_n'],
            'omnibus_stat': stats_result['omnibus_stat'],
            'omnibus_p': stats_result['omnibus_p'],
        }
        saved_paths['summary'] = save_json(summary, output_dir, f"{group_name}_smi_stats_summary.json")

        fig = stats_result.get('fig')
        if fig is not None:
            saved_paths['figure'] = save_figure_png(fig, output_dir, f"{group_name}_smi_comparison.png")

    return saved_paths


def save_all_comparison_outputs(output_dir, all_group_dfs, smi_comparison_results):
    """
    Loop save_comparison_group_outputs over every group in all_group_dfs.

    Parameters
    ----------
    output_dir : str
    all_group_dfs : dict
        {group_name: df}, from Function 4.4's run_all_comparisons.
    smi_comparison_results : dict
        {group_name: result with 'fig'}, from Function 4.7's
        run_all_group_comparisons.

    Returns
    -------
    saved_paths_by_group : dict
        {group_name: save_comparison_group_outputs(...) result}.
    """
    saved_paths_by_group = {}
    for group_name, df in all_group_dfs.items():
        stats_result = smi_comparison_results.get(group_name)
        saved_paths_by_group[group_name] = save_comparison_group_outputs(
            output_dir, group_name, df, stats_result=stats_result
        )
    print(f"\nSaved outputs for {len(saved_paths_by_group)} group(s) to {output_dir}")
    return saved_paths_by_group


def save_reliability_calibration_outputs(output_dir, calib_df, coeffs, fig):
    """
    Save Function 4.5's outputs: the calibration table (CSV), fit
    coefficients (JSON), and the scatter plot (PNG).
    """
    return {
        'table': save_dataframe_csv(calib_df, output_dir, "reliability_calibration.csv"),
        'coeffs': save_json({'slope': coeffs[0], 'intercept': coeffs[1]},
                             output_dir, "reliability_calibration_fit.json"),
        'figure': save_figure_png(fig, output_dir, "reliability_calibration_plot.png"),
    }


def save_reliability_component_outputs(output_dir, component_calib_df, component_results):
    """
    Save Function 4.6's outputs: the component calibration table (CSV)
    and each fraction column's regression summary (one .txt per column --
    these are printed statsmodels tables, not figures).
    """
    saved_paths = {'table': save_dataframe_csv(component_calib_df, output_dir, "reliability_components.csv")}

    os.makedirs(output_dir, exist_ok=True)
    saved_paths['regressions'] = {}
    for fraction_col, model_result in component_results.items():
        txt_path = os.path.join(output_dir, f"reliability_regression_{fraction_col}.txt")
        with open(txt_path, 'w') as f:
            f.write(str(model_result.summary()))
        print(f"Saved -> {txt_path}")
        saved_paths['regressions'][fraction_col] = txt_path

    return saved_paths


def save_all_phase4_outputs(output_dir, all_group_dfs, smi_comparison_results,
                             calib_df=None, coeffs=None, calib_fig=None,
                             component_calib_df=None, component_results=None):
    """
    Save everything Phase 4 generates for one animal in one call. See
    markdown above for the full file list.

    Parameters
    ----------
    output_dir : str
        e.g. os.path.join(ANIMAL_DIR, 'Phase4_SessionComparison_Results').
    all_group_dfs : dict
    smi_comparison_results : dict
        Must carry a 'fig' key per group -- true automatically now that
        Function 4.7's run_all_group_comparisons stores it.
    calib_df, coeffs, calib_fig : optional
        From Function 4.5, if you ran it.
    component_calib_df, component_results : optional
        From Function 4.6, if you ran it.

    Returns
    -------
    saved_paths : dict
    """
    saved_paths = {'comparisons': save_all_comparison_outputs(output_dir, all_group_dfs, smi_comparison_results)}

    if calib_df is not None and coeffs is not None and calib_fig is not None:
        saved_paths['reliability_calibration'] = save_reliability_calibration_outputs(
            output_dir, calib_df, coeffs, calib_fig
        )

    if component_calib_df is not None and component_results is not None:
        saved_paths['reliability_components'] = save_reliability_component_outputs(
            output_dir, component_calib_df, component_results
        )

    print(f"\nAll Phase 4 outputs saved under: {output_dir}")
    return saved_paths


In [21]:
# --- Save everything Phase 4 generated for this animal ---
OUTPUT_DIR = os.path.join(TEST_ANIMAL_DIR, 'Phase4_SessionComparison_Results')

saved_paths = save_all_phase4_outputs(
    OUTPUT_DIR, all_group_dfs, smi_comparison_results,
    calib_df=calib_df, coeffs=calib_coeffs, calib_fig=fig,
    component_calib_df=component_calib_df, component_results=component_results,
)


Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\Phase4_SessionComparison_Results\dcz1_comparison_table.csv
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\Phase4_SessionComparison_Results\dcz1_smi_pairwise_stats.csv
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\Phase4_SessionComparison_Results\dcz1_smi_stats_summary.json
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\Phase4_SessionComparison_Results\dcz1_smi_comparison.png
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\Phase4_SessionComparison_Results\dcz2_comparison_table.csv
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\Phase4_SessionComparison_Results\dcz2_smi_pairwise_stats.csv
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD\Phase4_SessionComparison_Results\dcz2_smi_stats_summary.json
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1pr